# Data Wrangling — Falcon 9 First-Stage Landing Dataset

This notebook documents the cleaning and feature-preparation steps for the frozen IBM course Falcon 9 dataset. It preserves the course Class target (1 = successful first-stage landing, 0 = unsuccessful first-stage landing).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA=Path('data/dataset_part_2.csv')
OUT=Path('outputs/wrangling')
OUT.mkdir(parents=True,exist_ok=True)
if not DATA.exists(): raise FileNotFoundError(DATA)
df=pd.read_csv(DATA)
df.columns=[str(c).strip() for c in df.columns]
required={'FlightNumber','Date','PayloadMass','Orbit','LaunchSite','Class'}
missing=required-set(df.columns)
if missing: raise ValueError(f'Missing required fields: {sorted(missing)}')
print('Initial shape:',df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame('Missing values'))

Initial shape: (90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


,Missing values
LandingPad,26
FlightNumber,0
Date,0
BoosterVersion,0
Orbit,0
PayloadMass,0
Outcome,0
Flights,0
GridFins,0
LaunchSite,0


## 1. Validate data types and target definition

Convert dates and payload mass to appropriate types. Records without a valid target are excluded; unknown landing outcomes are not guessed.

In [2]:
df['Date']=pd.to_datetime(df['Date'],errors='coerce')
df['FlightNumber']=pd.to_numeric(df['FlightNumber'],errors='coerce')
df['PayloadMass']=pd.to_numeric(df['PayloadMass'],errors='coerce')
df['Class']=pd.to_numeric(df['Class'],errors='coerce')
df=df.dropna(subset=['FlightNumber','Date','Class']).copy()
df['Class']=df['Class'].astype(int)
if not set(df['Class'].unique()).issubset({0,1}): raise ValueError('Class must be binary 0/1')
print('Validated shape:',df.shape)
display(df.dtypes.to_frame('Data type'))

Validated shape: (90, 18)


,Data type
FlightNumber,int64
Date,datetime64[ns]
BoosterVersion,object
PayloadMass,float64
Orbit,object
LaunchSite,object
Outcome,object
Flights,int64
GridFins,bool
Reused,bool


## 2. Treat missing payload mass

Payload mass is numeric. For this exercise, missing payload values are replaced with the observed mean. The imputation value is printed and retained for transparency.

In [3]:
payload_mean=df['PayloadMass'].mean()
missing_payload=int(df['PayloadMass'].isna().sum())
df['PayloadMass']=df['PayloadMass'].fillna(payload_mean)
print('Missing payload values before imputation:',missing_payload)
print('Imputation mean (kg):',round(float(payload_mean),3))
print('Missing payload values after imputation:',int(df['PayloadMass'].isna().sum()))

Missing payload values before imputation: 0
Imputation mean (kg): 6104.959
Missing payload values after imputation: 0


## 3. Select model features and encode categorical variables

Use the course feature set where available. One-hot encode categorical fields and preserve the Class target separately. Avoid including raw outcome text or the Class label among the predictors.

In [4]:
candidate_features=['FlightNumber','PayloadMass','Orbit','LaunchSite','Flights','GridFins','Reused','Legs','LandingPad','Block','ReusedCount','Serial']
features=[c for c in candidate_features if c in df.columns]
if not features: raise ValueError('No expected model feature columns found')
X=df[features].copy()
y=df['Class'].copy()
categorical=X.select_dtypes(include=['object','category','string','bool']).columns.tolist()
X_encoded=pd.get_dummies(X,columns=categorical,dtype=float)
X_encoded=X_encoded.replace([np.inf,-np.inf],np.nan)
for c in X_encoded.columns:
    if X_encoded[c].isna().any(): X_encoded[c]=X_encoded[c].fillna(X_encoded[c].median())
print('Selected base features:',features)
print('Categorical and boolean variables encoded:',categorical)
print('Encoded feature matrix shape:',X_encoded.shape)
print('Target shape:',y.shape)
display(X_encoded.head())

official_path=Path('data/dataset_part_3.csv')
if official_path.exists():
    official=pd.read_csv(official_path)
    print('Official course feature matrix shape:',official.shape)
    print('Recreated feature count matches official matrix:',X_encoded.shape[1]==official.shape[1])
    print('Recreated feature names match official matrix:',set(X_encoded.columns)==set(official.columns))
    if len(official)!=len(X_encoded): raise ValueError('Official and recreated feature matrices have different row counts.')
    if X_encoded.shape[1]!=official.shape[1] or set(X_encoded.columns)!=set(official.columns):
        raise ValueError('Recreated matrix does not match the official IBM feature matrix; inspect one-hot encoded fields.')

Selected base features: ['FlightNumber', 'PayloadMass', 'Orbit', 'LaunchSite', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial']
Categorical and boolean variables encoded: ['Orbit', 'LaunchSite', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Serial']
Encoded feature matrix shape: (90, 83)
Target shape: (90,)


,FlightNumber,PayloadMass,Flights,Block,ReusedCount,Orbit_ES-L1,Orbit_GEO,Orbit_GTO,Orbit_HEO,Orbit_ISS,...,Serial_B1048,Serial_B1049,Serial_B1050,Serial_B1051,Serial_B1054,Serial_B1056,Serial_B1058,Serial_B1059,Serial_B1060,Serial_B1062
0,1,6104.959412,1,1.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,525.000000,1,1.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,677.000000,1,1.0,0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,500.000000,1,1.0,0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,3170.000000,1,1.0,0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Official course feature matrix shape: (90, 83)
Recreated feature count matches official matrix: True
Recreated feature names match official matrix: True


The feature matrix is validated against the official IBM course dataset_part_3.csv. Boolean fields are one-hot encoded as well as text categories, so all dummy columns are preserved. The model notebook uses the official feature matrix to keep the predictive analysis aligned with the course lab.


In [5]:
df.to_csv(OUT/'dataset_part_2_cleaned.csv',index=False)
X_encoded.to_csv(OUT/'dataset_part_3_recreated.csv',index=False)
y.to_csv(OUT/'landing_target_class.csv',index=False,header=['Class'])
quality=pd.DataFrame({
    'Rows':[len(df)],
    'Base feature count':[len(features)],
    'Encoded feature count':[X_encoded.shape[1]],
    'Missing payload after imputation':[int(df['PayloadMass'].isna().sum())],
    'Missing target after cleaning':[int(df['Class'].isna().sum())],
    'Duplicate rows':[int(df.duplicated().sum())]
})
display(quality)
print('Outputs saved under:',OUT.resolve())

,Rows,Base feature count,Encoded feature count,Missing payload after imputation,Missing target after cleaning,Duplicate rows
0,90,12,83,0,0,0


Outputs saved under: /home/runner/work/Data-Science-Capstone-Project-Report/Data-Science-Capstone-Project-Report/outputs/wrangling
